In [ ]:
import os
import pandas as pd
import numpy as np

from PIL import Image
import matplotlib.pylab as plt
import cv2

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
import kagglehub

# Download latest version
gun_path = kagglehub.dataset_download("atulyakumar98/gundetection")

print("Path to dataset files:", gun_path)

Using Colab cache for faster access to the 'gundetection' dataset.
Path to dataset files: /kaggle/input/gundetection


In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip

--2026-09-15 06:58:05--  http://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.183.79, 16.15.212.179, 16.15.238.151, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.183.79|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 815585330 (778M) [application/zip]
Saving to: ‘val2017.zip’

val2017.zip         100%[===================>] 777.80M  98.3MB/s    in 12s     

2026-09-15 06:58:18 (64.5 MB/s) - ‘val2017.zip’ saved [815585330/815585330]



In [ ]:
!unzip -q val2017.zip -d ./dataset

In [ ]:
import os
os.rename("./dataset/val2017", "./dataset/no_gun")

In [ ]:
! rm -r val2017.zip

In [ ]:
os.mkdir("./dataset/gun")

In [ ]:
import shutil

In [ ]:
for name in os.listdir(gun_path):
  if name.strip().endswith('.jpg'):
    src_path = os.path.join(gun_path, name)
    dst_path = os.path.join("./dataset/gun", name)
    shutil.copy(src_path, dst_path)

In [ ]:
len(os.listdir("./dataset/gun")), len(os.listdir("./dataset/no_gun"))

(3000, 5000)

In [ ]:
[os.remove(os.path.join("./dataset/no_gun", name)) for name in os.listdir("./dataset/no_gun")[:2000]]

print("Gun files" , len(os.listdir("./dataset/gun")))
print("No gun files" , len(os.listdir("./dataset/no_gun")))

Gun files 3000
No gun files 3000


In [ ]:
# gun_labels = [1]*len(os.listdir("./dataset/gun"))
# no_gun_labels = [0]*len(os.listdir("./dataset/no_gun"))

In [ ]:
# labels = gun_labels + no_gun_labels

In [ ]:
# data = []
# for name in os.listdir("./dataset/gun"):
#   image = Image.open(os.path.join("./dataset/gun", name))
#   image = image.resize((128,128))
#   image = image.convert("RGB")
#   arr = np.array(image)
#   data.append(arr)

# for name in os.listdir("./dataset/no_gun"):
#   image = Image.open(os.path.join("./dataset/no_gun", name))
#   image = image.resize((128,128))
#   image = image.convert("RGB")
#   arr = np.array(image)
#   data.append(arr)

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

batch_size = 32
img_size = (128,128)

train_ds = image_dataset_from_directory(
  "./dataset",
  validation_split=0.2,
  subset="training",
  seed=42,
  image_size=img_size,
  batch_size=batch_size,
  label_mode='binary',
  class_names = ['no_gun', 'gun'])

temp_val_ds = image_dataset_from_directory(
  "./dataset",
  validation_split=0.2,
  subset="validation",
  seed=42,
  image_size=img_size,
  batch_size=batch_size,
  label_mode = 'binary',
  class_names = ['no_gun', 'gun'])


val_batches = tf.data.experimental.cardinality(temp_val_ds) // 2

val_ds = temp_val_ds.take(val_batches)
test_ds = temp_val_ds.skip(val_batches)

print(f"Train Batches: {len(train_ds)}")
print(f"Validation Batches: {len(val_ds)}")
print(f"Test Batches: {len(test_ds)}")

Found 6000 files belonging to 2 classes.
Using 4800 files for training.
Found 6000 files belonging to 2 classes.
Using 1200 files for validation.
Train Batches: 150
Validation Batches: 19
Test Batches: 19


In [ ]:
for images, labels in train_ds.take(1):
    print("Images Data Type:", type(images)) # tf.Tensor (Not a plain list)
    print("Images Shape:", images.shape)      # Output: (32, 128, 128, 3)
    print("Labels Shape:", labels.shape)      # Output: (32, 1) or (32,)

Images Data Type: <class 'tensorflow.python.framework.ops.EagerTensor'>
Images Shape: (32, 128, 128, 3)
Labels Shape: (32, 1)


In [65]:
model = Sequential()

input = keras.Input(shape=(128,128,3))
x = layers.Rescaling(1./255)(input)
x = layers.Conv2D(filters=16, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Flatten()(x)
x = layers.Dense(units=128, activation="relu")(x)
output = layers.Dense(units=2, activation="sigmoid")(x)

model = keras.Model(inputs=input, outputs=output)

In [66]:
model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 687,650 (2.62 MB)

 Trainable params: 687,650 (2.62 MB)

 Non-trainable params: 0 (0.00 B)

In [70]:
callbacks = [
 keras.callbacks.ModelCheckpoint(
 filepath="gun_detection_from_scratch.keras",
 save_best_only=True,
 monitor="val_loss")
]

In [79]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

In [80]:
history = model.fit(train_ds,
          validation_data=val_ds,
          epochs=10,
          callbacks=callbacks)

Epoch 1/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 112s 723ms/step - accuracy: 0.9660 - loss: 0.0936 - val_accuracy: 0.7911 - val_loss: 0.7431
Epoch 2/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 98s 649ms/step - accuracy: 0.9835 - loss: 0.0490 - val_accuracy: 0.7993 - val_loss: 1.2654
Epoch 3/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 140s 640ms/step - accuracy: 0.9583 - loss: 0.1225 - val_accuracy: 0.8322 - val_loss: 0.6880
Epoch 4/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 94s 628ms/step - accuracy: 0.9908 - loss: 0.0319 - val_accuracy: 0.8043 - val_loss: 0.9932
Epoch 5/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 148s 668ms/step - accuracy: 0.9937 - loss: 0.0203 - val_accuracy: 0.8191 - val_loss: 1.0112
Epoch 6/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 99s 659ms/step - accuracy: 0.9769 - loss: 0.0616 - val_accuracy: 0.7961 - val_loss: 0.8441
Epoch 7/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 99s 657ms/step - accuracy: 0.9873 - loss: 0.0334 - val_accuracy: 0.8191 - val_loss: 0.9388
Epoch 8/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 106s 704ms/step - accuracy: 0.9775 - los

In [82]:
model.evaluate(test_ds)  # its completely overfitting

19/19 ━━━━━━━━━━━━━━━━━━━━ 8s 307ms/step - accuracy: 0.8024 - loss: 0.9204


[0.9203674793243408, 0.8023648858070374]

In [83]:
# DATA AUGMENTATION
data_augmentation = keras.Sequential(
 [
          layers.RandomFlip("horizontal"),
          layers.RandomRotation(0.1),
          layers.RandomZoom(0.2),
 ]
)

In [85]:
model = Sequential()

input = keras.Input(shape=(128,128,3))
X = data_augmentation(input)
x = layers.Rescaling(1./255)(input)
x = layers.Conv2D(filters=16, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(units=128, activation="relu")(x)
output = layers.Dense(units=2, activation="sigmoid")(x)

model = keras.Model(inputs=input, outputs=output)

In [90]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

my_callbacks = [
    # 1. Disk par best model save karega
    ModelCheckpoint(
        filepath="best_gun_detector.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min"
    ),
    # 2. Overfitting hone par training roke ga aur RAM mein best weights restore karega
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        mode="min"
    )
]

In [91]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

In [92]:
history1 = model.fit(train_ds,
          validation_data=val_ds,
          epochs=100,
          callbacks=my_callbacks)

Epoch 1/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 108s 698ms/step - accuracy: 0.8096 - loss: 0.4154 - val_accuracy: 0.8289 - val_loss: 0.3834
Epoch 2/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 131s 875ms/step - accuracy: 0.8235 - loss: 0.3918 - val_accuracy: 0.8257 - val_loss: 0.3818
Epoch 3/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 106s 697ms/step - accuracy: 0.8254 - loss: 0.3757 - val_accuracy: 0.8306 - val_loss: 0.3832
Epoch 4/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 130s 624ms/step - accuracy: 0.8533 - loss: 0.3423 - val_accuracy: 0.8289 - val_loss: 0.3811
Epoch 5/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 139s 604ms/step - accuracy: 0.8533 - loss: 0.3318 - val_accuracy: 0.8355 - val_loss: 0.3915
Epoch 6/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 90s 596ms/step - accuracy: 0.8608 - loss: 0.3175 - val_accuracy: 0.8470 - val_loss: 0.4030
Epoch 7/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 94s 624ms/step - accuracy: 0.8635 - loss: 0.3076 - val_accuracy: 0.8520 - val_loss: 0.4130
Epoch 8/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 98s 654ms/step - accuracy: 0.8

In [97]:
model.evaluate(train_ds)

150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 225ms/step - accuracy: 0.8715 - loss: 0.2983


[0.2982608377933502, 0.8714583516120911]

In [94]:
model.evaluate(test_ds)

19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 204ms/step - accuracy: 0.8378 - loss: 0.3911


[0.39110586047172546, 0.837837815284729]

In [95]:
train_ped = model.predict(train_ds)

150/150 ━━━━━━━━━━━━━━━━━━━━ 42s 271ms/step


In [112]:
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array

class_names = ['gun', 'no_gun']

def pred_image(img_path):
    try:
        # 1. Image load aur array conversion
        img = load_img(img_path, target_size=(128, 128))
        img_array = img_to_array(img)

        # 2. Rescaling check: Agar model input layer mein Rescaling nahi hai
        # img_array = img_array / 255.0

        # 3. Batch dimension
        img_array = np.expand_dims(img_array, axis=0)

        # 4. Prediction
        pred_prob = model.predict(img_array, verbose=0)[0][0]

        # 5. Output logic
        if pred_prob > 0.5:
            predicted_class = class_names[1]
            confidence = pred_prob * 100
        else:
            predicted_class = class_names[0]
            confidence = (1 - pred_prob) * 100

        return f"Prediction: {predicted_class} (Confidence: {confidence:.2f}%)"

    except Exception as e:
        # Exact Error Text Screen Par Display Hoga
        return f"Python Error Details: {str(e)}"

# Gradio Setup
demo = gr.Interface(
    fn=pred_image,
    inputs=gr.Image(type="filepath"),
    outputs="text",
    title="Gun Detection Classifier"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc5372dafc94ba355e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
